# Spark Exercises 07 — UDFs, Built-ins & Performance

This chapter is about what Spark **lets you do** and what it **punishes you for**. You *can* write a plain Python UDF — but it becomes a black box that Catalyst cannot optimize and that serializes every row to Python. You'll see the native alternative, the vectorized `pandas_udf` middle ground, and the partition controls (`repartition` / `coalesce` / `cache`) that decide how fast a job runs on 250,000 rows.

**Data files:** `data/web_events.csv` (250k rows), `data/orders.csv`

---

> **Solutions version** — try it yourself first from `Exercises.ipynb`.

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


### 2. Read `data/web_events.csv` (250,000 rows) into `web` (`header=True`, `inferSchema=True`) and `count()` it.

In [2]:
web = spark.read.csv("data/web_events.csv", header=True, inferSchema=True)
web.count()

250000

### 3. How many **partitions** is `web` split into? (`web.rdd.getNumPartitions()`)

In [3]:
web.rdd.getNumPartitions()

4

### 4. **The tempting way — a Python UDF.** Write a Python function that labels `duration_ms` as `'fast'` (< 1000), `'medium'` (< 5000) or `'slow'`, wrap it with `F.udf`, and apply it. Show 5 rows. ⚠️ This works, but every row is shipped to a Python worker and Catalyst sees a black box.

In [4]:
from pyspark.sql.types import StringType

def speed_label(ms):
    if ms < 1000:
        return "fast"
    elif ms < 5000:
        return "medium"
    return "slow"

speed_udf = F.udf(speed_label, StringType())
web.withColumn("speed", speed_udf("duration_ms")) \
   .select("page", "duration_ms", "speed").show(5)

+---------+-----------+------+
|     page|duration_ms| speed|
+---------+-----------+------+
|    /cart|       7443|  slow|
|/checkout|       4042|medium|
| /account|      18441|  slow|
| /product|      18522|  slow|
|    /blog|        463|  fast|
+---------+-----------+------+
only showing top 5 rows



### 5. **The Spark way — native built-ins.** Reproduce the exact same labelling with `F.when(...).otherwise(...)`. This stays inside the JVM, is vectorized, and Catalyst can optimize it. Show 5 rows.

In [5]:
native = web.withColumn(
    "speed",
    F.when(F.col("duration_ms") < 1000, "fast")
     .when(F.col("duration_ms") < 5000, "medium")
     .otherwise("slow"),
)
native.select("page", "duration_ms", "speed").show(5)

+---------+-----------+------+
|     page|duration_ms| speed|
+---------+-----------+------+
|    /cart|       7443|  slow|
|/checkout|       4042|medium|
| /account|      18441|  slow|
| /product|      18522|  slow|
|    /blog|        463|  fast|
+---------+-----------+------+
only showing top 5 rows



### 6. Confirm both approaches agree: count rows **per `speed`** using the native version.

In [6]:
native.groupBy("speed").count().orderBy(F.col("count").desc()).show()

+------+------+
| speed| count|
+------+------+
|  slow|208473|
|medium| 33238|
|  fast|  8289|
+------+------+



### 7. **The middle ground — a vectorized `pandas_udf`.** When you truly need Python, a `pandas_udf` processes a whole batch as a pandas Series (using Arrow), far faster than a row-by-row UDF. Write one that converts `duration_ms` to seconds and apply it. Show 5 rows.

In [7]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf("double")
def to_seconds(ms: pd.Series) -> pd.Series:
    return ms / 1000.0

web.withColumn("seconds", to_seconds("duration_ms")) \
   .select("duration_ms", "seconds").show(5)

+-----------+-------+
|duration_ms|seconds|
+-----------+-------+
|       7443|  7.443|
|       4042|  4.042|
|      18441| 18.441|
|      18522| 18.522|
|        463|  0.463|
+-----------+-------+
only showing top 5 rows



### 8. **`repartition`** reshuffles the data into a chosen number of partitions (a full shuffle). Repartition `web` into 16 partitions and confirm the new count.

In [8]:
web16 = web.repartition(16)
web16.rdd.getNumPartitions()

16

### 9. **`coalesce`** only *reduces* partitions, and avoids a full shuffle — ideal right before writing output. Coalesce `web` down to 2 partitions and confirm.

In [9]:
web.coalesce(2).rdd.getNumPartitions()

2

### 10. A realistic aggregation: **average `duration_ms` and event count per `page`**, slowest pages first (round the average).

In [10]:
web.groupBy("page") \
   .agg(F.round(F.avg("duration_ms"), 1).alias("avg_ms"),
        F.count("*").alias("events")) \
   .orderBy(F.col("avg_ms").desc()).show()

+---------+-------+------+
|     page| avg_ms|events|
+---------+-------+------+
| /product|15073.4| 31449|
|    /home|15070.0| 31339|
|    /cart|15058.4| 31364|
| /account|15057.9| 31004|
|    /help|15013.4| 31097|
|/checkout|15012.2| 31540|
|    /blog|15006.9| 31196|
|  /search|15002.8| 31011|
+---------+-------+------+



### 11. If you will reuse a filtered DataFrame several times, **`cache`** it so Spark does not recompute it each time. Cache the `submit` events, materialize with `count()`, then check `storageLevel`.

In [11]:
submits = web.filter(F.col("event_type") == "submit").cache()
print("rows:", submits.count())
print(submits.storageLevel)

rows: 12407
Disk Memory Deserialized 1x Replicated


### 12. **`count()` scans everything; `take()` does not.** `count()` the whole table (an action over all 250k rows), then `take(3)` — which stops as soon as it has 3 rows. Notice `take` returns a small Python list.

In [12]:
print("total rows:", web.count())
web.take(3)

total rows: 250000


[Row(event_ts=datetime.datetime(2024, 6, 7, 16, 32, 21), user_id='CUST-0320', page='/cart', event_type='view', duration_ms=7443, country='JP'),
 Row(event_ts=datetime.datetime(2024, 6, 7, 19, 16, 58), user_id='CUST-0045', page='/checkout', event_type='click', duration_ms=4042, country='DE'),
 Row(event_ts=datetime.datetime(2024, 6, 19, 12, 16, 21), user_id='CUST-0276', page='/account', event_type='click', duration_ms=18441, country='AU')]